# CS781 Phase 1 — Notebook 2 of 4, VERSION 2: ENCODER-ONLY Transformer (ModernBERT-style, masked-language modelling)

**Why a version 2:** the first run finished well (final validation loss **1.747**: Supreme Court 1.591, high courts 1.683, acts 1.524, tribunals 2.192) but its checkpoint was lost because it was never saved outside the session. Same model, same recipe and **same general-purpose data mix** (30% Supreme Court, 30% tribunals, 35% high courts, 5% acts: this is a foundation model, adapted to tasks such as CJPE later by fine-tuning), plus these changes:

| change | why |
|---|---|
| **cell 9 pushes the model to a private Kaggle dataset automatically** (needs the Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached and Internet ON; never asks questions, so it works in *Save & Run All*) | the loss of the first run; without credentials it only warns, then download the three files yourself |
| `HOURS = 3.5` instead of 5.0 (about 1,750 steps, 0.23B tokens) | the first run took 5.9 h. Expect a final masked-token loss nearer 2.0-2.1 than 1.75 (my estimate: the curve was still improving, and about 0.2 of the gain came from the final learning-rate decay, which a shorter run still gets). Set `HOURS = 5.0` in cell 1 to match the first run |
| background data prefetch (`PREFETCH = True` in the trainer cell) + a speed profile (cell 6b) | the first run held 18.6k tokens/s against 21.7k measured in the smoke test (a 14% loss, the same as in the GPT-2 run). The profile shows whether batch building is the cause; if the GPU clocks are throttled instead, nothing in the code can fix it |

Everything else (model, masking, optimiser, schedule, validation, data mix) is unchanged and was stable in the first run: gradient norm 0.4-0.6, no spikes, validation improved at every check.

**Model:** `ModernBertForMaskedLM`, 12 layers x 768, GeGLU, RoPE, local + global attention, 85.4M parameters. **Objective:** 30% masking (80% `[MASK]`, 10% random, 10% unchanged). **Tokenizer:** your 32,000 legal BPE plus `[MASK]` (id 32000).

## Run order
1 setup -> 2 data -> 3/4 architecture -> 5 trainer -> **6 smoke test (must pass)** -> 6b speed profile (optional, 2 min) -> 7 the run -> 8 export -> 9 automatic push.
**Safest way to run it:** attach the two Secrets, turn Internet on, run cells 1-6 once interactively to check the smoke test, then Save Version -> **Save & Run All**. The finished notebook keeps its output files, and cell 9 also pushes them to your dataset.

## What was tested
On CPU with a tiny version: every cell, the prefetch (it yields exactly the same batches as the plain stream), the profile cell, and the push cell with a stand-in `kaggle` command (no credentials -> a warning and no crash; with credentials -> upload, and a new version when the dataset exists). **Not tested:** GPU speed and the real upload from inside a committed run.


In [ ]:
# ================= CELL 1 — setup, paths, tokenizer (+[MASK]), configuration =================
ARCH = "modernbert"
import os, sys, re, json, math, time, random, glob, gzip, gc, copy, contextlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch, torch.nn as nn

BASE    = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus, manifests, tokenizer
P1_CKPT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"    # only used for gpt2 with init='warm3000'
OUT = Path("/kaggle/working/pretrain_modernbert"); OUT.mkdir(parents=True, exist_ok=True)
assert os.path.exists(BASE), f"missing: {BASE}"
REQUIRE_GPU, TINY = True, False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_GPU: assert device.type == "cuda", "pretraining needs a GPU"
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8      # T4 (7.5) has no native bf16 -> fp16 + GradScaler
AMP = device.type == "cuda"
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| precision:", ("bf16" if USE_BF16 else "fp16 + GradScaler") if AMP else "fp32")

# ---- tokenizer: the original 32,000-token legal BPE plus ONE new special token, [MASK] (id 32000). Identical in all four notebooks. ----
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json"); n0 = tok.get_vocab_size()
PAD, UNK, BOS, EOS = (tok.token_to_id(t) for t in ("[PAD]", "[UNK]", "[BOS]", "[EOS]")); assert None not in (PAD, UNK, BOS, EOS)
if tok.token_to_id("[MASK]") is None: tok.add_special_tokens(["[MASK]"])
MASK = tok.token_to_id("[MASK]"); V = tok.get_vocab_size(); assert MASK == n0 and V == n0 + 1, (MASK, V, n0)
tok.save(str(OUT / "legal_bpe_32k_mask.json"))
print(f"tokenizer: {n0} -> {V} tokens | PAD/UNK/BOS/EOS/MASK = {PAD}/{UNK}/{BOS}/{EOS}/{MASK} | saved {OUT/'legal_bpe_32k_mask.json'} (attach this file to the Phase 2/3 notebooks)")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
SEQ = 512                               # pre-training block length (downstream windows are <= 512 tokens)
COMMON = dict(lr_min_frac=0.1, warmup=100, wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144, min_words=50, seed=1234,
              domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05})     # downstream texts are judgments, not statutes
CFG_ARCH = dict(micro_bs=16, accum=16, lr_peak=6e-4, amp=True)      # tokens per optimizer step = micro_bs * SEQ * accum = 131,072
HOURS = 3.5       # suggested GPU-hours for this model (change freely)


In [ ]:
# ================= CELL 2 — streaming corpus: held-out split, domain-weighted shuffled mix, packing WITHOUT padding =================
def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")
def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]               # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words):
    files = list(shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):                              # shard files are ordered, so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> [BOS] tokens [EOS] -> packed into [micro_bs, SEQ] blocks with NO padding"""
    def __init__(self, tokenizer, micro_bs, seq, domain_w, min_words, seed, skip):
        self.tok, self.B, self.T, self.rng = tokenizer, micro_bs, seq, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, min_words), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [domain_w[d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = [BOS] + enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        self._refill(self.B * self.T); x = torch.tensor(self.buf[:self.B * self.T], dtype=torch.long).view(self.B, self.T); del self.buf[:self.B * self.T]; return x

def build_val_blocks(per_domain_tokens, cache):
    """fixed held-out blocks per domain, from validation-manifest documents only, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    per = per_domain_tokens // SEQ; need = per * SEQ; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = [BOS] + tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, SEQ)
    torch.save(val, cache); return val
VAL = build_val_blocks(COMMON["eval_tokens_per_domain"], OUT / f"val_blocks_{SEQ}.pt")
print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


In [ ]:
# ================= CELL 3/4 — ARCHITECTURE: encoder-only Transformer (ModernBERT-style), bidirectional masked-language modelling =================
from transformers import ModernBertConfig, ModernBertForMaskedLM
SPEC = dict(hidden_size=768, intermediate_size=1152, num_hidden_layers=12, num_attention_heads=12, local_attention=128, global_attn_every_n_layers=3)   # RoPE, GeGLU, no biases, local+global attention
SPECIAL = [PAD, UNK, BOS, EOS]

def build_model(tiny=False, init="scratch"):
    """tiny=True is only used by the unit tests"""
    s = SPEC if not tiny else dict(hidden_size=64, intermediate_size=96, num_hidden_layers=4, num_attention_heads=4, local_attention=16, global_attn_every_n_layers=2)
    cfg = ModernBertConfig(vocab_size=V, max_position_embeddings=SEQ, pad_token_id=PAD, bos_token_id=BOS, eos_token_id=EOS, cls_token_id=BOS, sep_token_id=EOS,
                           attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False, **s)
    m = ModernBertForMaskedLM._from_config(cfg, attn_implementation="sdpa") if not tiny else ModernBertForMaskedLM(cfg)
    mc = dict(family="modernbert", vocab_size=V, context_length=SEQ, hidden_size=s["hidden_size"], intermediate_size=s["intermediate_size"], num_layers=s["num_hidden_layers"],
              num_heads=s["num_attention_heads"], local_attention=s["local_attention"], global_attn_every_n_layers=s["global_attn_every_n_layers"])
    m.config.use_cache = False; n = sum(p.numel() for p in m.parameters())
    if not tiny: assert 50e6 <= n <= 300e6, f"{n:,} parameters is outside the 50M-300M Phase 1 range"
    return (m if tiny else m.to(device)), mc, n

# ---- objective: BERT-style masking. 30% of the non-special tokens are selected; 80% -> [MASK], 10% -> a random token, 10% unchanged ----
def mlm_mask(x, p=0.30, gen=None):
    sel = torch.rand(x.shape, generator=gen, device=x.device) < p
    for s_ in SPECIAL: sel &= (x != s_)
    labels = torch.where(sel, x, torch.full_like(x, -100))
    r = torch.rand(x.shape, generator=gen, device=x.device); inp = x.clone()
    inp[sel & (r < 0.8)] = MASK
    rnd = sel & (r >= 0.8) & (r < 0.9); inp[rnd] = torch.randint(4, MASK, x.shape, generator=gen, device=x.device)[rnd]
    return inp, labels
def forward_loss(model, x, rng=None, gen=None):
    inp, labels = mlm_mask(x, 0.30, gen); return model(input_ids=inp, attention_mask=torch.ones_like(inp), labels=labels).loss
def val_item(x, gen, rng): return mlm_mask(x, 0.30, gen)
def eval_loss(model, it): return model(input_ids=it[0], attention_mask=torch.ones_like(it[0]), labels=it[1]).loss


In [ ]:
# ================= CELL 5 — evaluation + trainer (warmup-stable-decay, fp16/bf16, resume, per-domain validation) =================
def amp_ctx(enabled=True):
    if not (AMP and enabled): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)

_VALB = {}
def val_batches(bs=16, max_batches=None):
    """fixed, deterministic validation inputs (any masking / corruption uses a fixed seed, so every evaluation is comparable)"""
    key = (bs, max_batches)
    if key not in _VALB:
        gen = torch.Generator().manual_seed(123); rng = np.random.default_rng(123); out = {}
        for d, blocks in VAL.items():
            items = []
            for i in range(0, len(blocks), bs):
                if max_batches and len(items) >= max_batches: break
                items.append(val_item(blocks[i:i + bs], gen, rng))
            out[d] = items
        _VALB[key] = out
    return _VALB[key]

@torch.no_grad()
def evaluate(model, amp=True, bs=16, max_batches=None):
    model.eval(); res = {}
    for d, items in val_batches(bs, max_batches).items():
        tot = 0.0
        for it in items:
            it = [t.to(device) for t in it]
            with amp_ctx(amp): loss = eval_loss(model, it)
            tot += float(loss)
        res[d] = tot / max(1, len(items))
    res["overall"] = float(np.mean([res[d] for d in VAL])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())

import threading, queue
PREFETCH = True        # build the next batches on a background thread while the GPU computes (the tokenizer runs in Rust and releases the GIL)
class Prefetch:
    def __init__(self, stream, depth=6):
        self.stream, self.q, self.err, self._stop = stream, queue.Queue(maxsize=depth), None, False
        self.t = threading.Thread(target=self._run, daemon=True); self.t.start()
    def _run(self):
        try:
            while not self._stop: self.q.put(self.stream.next_batch())
        except Exception as e: self.err = e; self.q.put(None)
    def next_batch(self):
        x = self.q.get()
        if x is None: raise self.err
        return x
    @property
    def dom_tokens(self): return self.stream.dom_tokens
    def close(self):
        self._stop = True
        try:
            while True: self.q.get_nowait()
        except queue.Empty: pass

def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.7 * T); lo = c["lr_peak"] * c["lr_min_frac"]
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (lo - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(cfg=None, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None, max_val_batches=None, keep_model=False, quiet=False, run=None):
    c = {**COMMON, **CFG_ARCH, **(cfg or {})}; c.setdefault("total_steps", 3000); c.setdefault("decay_start", None); c["accum"] = accum or c["accum"]
    rdir = OUT / (run or ARCH); rdir.mkdir(exist_ok=True); torch.manual_seed(c["seed"])
    model, mc, n_params = build_model(tiny=TINY, init=c.get("init", "scratch")); model.train()
    groups = [{"params": [p for p in model.parameters() if p.ndim >= 2], "weight_decay": c["wd"]}, {"params": [p for p in model.parameters() if p.ndim < 2], "weight_decay": 0.0}]
    try: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=(device.type == "cuda"))
    except Exception: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8)
    use_amp = AMP and c["amp"]; scaler = torch.amp.GradScaler("cuda", enabled=use_amp and not USE_BF16); step, tokens = 0, 0; ck_path = rdir / "latest.pt"
    if save and resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"[{ARCH}] resumed from step {step}"); del ck
    stream = PackedStream(tok, c["micro_bs"], SEQ, c["domain_w"], c["min_words"], c["seed"] + step, SKIP)
    if PREFETCH: stream = Prefetch(stream)
    rng = np.random.default_rng(c["seed"] + step); gen = torch.Generator(device=device).manual_seed(c["seed"] + step)
    T = c["micro_bs"] * SEQ * c["accum"]; end = steps if steps is not None else c["total_steps"]; log = open(rdir / "log.jsonl", "a"); t0 = time.time(); last_save = time.time()
    best, run_loss, nb, step_times, hist, bad = float("inf"), 0.0, 0, [], [], 0; ev = c["eval_every"]
    while step < end:
        ts = time.time(); lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        good = 0
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(use_amp): loss = forward_loss(model, x, rng=rng, gen=gen)
            if not torch.isfinite(loss): bad += 1; print(f"[{ARCH}] non-finite loss (total {bad}); skipping microbatch"); assert bad < 50, "too many non-finite losses: set amp=False in CFG_ARCH (cell 1)"; continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1; good += 1
        if good:
            scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update()
        else: gn = torch.tensor(float("nan"))
        opt.zero_grad(set_to_none=True); step += 1; tokens += T
        if device.type == "cuda": torch.cuda.synchronize()
        step_times.append(time.time() - ts)
        if step % 10 == 0:
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=T / float(np.median(step_times[-10:])), elapsed_min=(time.time() - t0) / 60); run_loss, nb = 0.0, 0
            log.write(json.dumps(rec) + "\n"); log.flush()
            if not quiet and (step % 50 == 0 or step <= 20): print(f"[{ARCH}] step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {rec['tok_s']/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % ev == 0 or step == end):
            r = evaluate(model, c["amp"], max_batches=max_val_batches); hist.append(dict(step=step, tokens=tokens, elapsed_min=(time.time() - t0) / 60, val=r)); log.write(json.dumps(dict(step=step, val=r)) + "\n"); log.flush()
            if not quiet: print(f"   [{ARCH}] VALIDATION step {step}: {fmt(r)}", flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print(f"   [{ARCH}] checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"[{ARCH}] time budget reached at step {step}"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    log.close(); tail = step_times[2:] if len(step_times) > 3 else step_times
    info = dict(history=hist, tok_s=float(T / np.median(tail)), params=n_params, steps=step, tokens=tokens, minutes=(time.time() - t0) / 60, model=model if keep_model else None,
                mix={d: v / max(1, sum(stream.dom_tokens.values())) for d, v in stream.dom_tokens.items()})
    if PREFETCH: stream.close()
    del opt, scaler; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return info

def plan_budget(tok_s, hours):
    c = {**COMMON, **CFG_ARCH}; per = c["micro_bs"] * SEQ * c["accum"]; steps = int(hours * 3600 * tok_s / per) // 10 * 10
    print(f"  at {tok_s/1e3:.1f}k tok/s, {hours} GPU-h = {steps} steps = {steps*per/1e9:.2f}B tokens"); return steps

def run_arch(hours=None, steps=None, run=None, **over):
    """probe the real throughput, size the schedule to the GPU-hour budget, then train (resumes automatically after an interruption)"""
    hours = HOURS if hours is None else hours
    probe = train(cfg=dict(init="scratch"), steps=8, accum=2, do_eval=False, save=False, resume=False, quiet=True, run="probe")
    steps = steps or plan_budget(probe["tok_s"], hours)
    cfg = dict(total_steps=steps, warmup=max(20, min(150, steps // 20)), decay_start=int(0.7 * steps), **over)
    print(f"[{ARCH}] {steps} steps for ~{hours} GPU-hours (measured {probe['tok_s']/1e3:.1f}k tok/s)")
    return train(cfg=cfg, run=run)


In [ ]:
# ================= CELL 6 — SMOKE TEST: 6 optimizer steps + validation (~2 min). Must pass before the real run. =================
info = train(cfg=dict(init="scratch"), steps=6, accum=2, do_eval=True, save=False, resume=False, max_val_batches=2, run="smoke")
v = info["history"][-1]["val"]; assert np.isfinite(v["overall"]), "non-finite validation loss - do not train before fixing (try amp=False in CFG_ARCH, cell 1)"
print(f"\n{ARCH}: {info['params']/1e6:.1f}M parameters | validation loss after 6 steps {v['overall']:.3f} | {info['tok_s']/1e3:.1f}k tokens/s on this GPU")
print("\nGPU-hour planning:")
for h in (2, 3, 5, 8): plan_budget(info["tok_s"], h)
del info; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= CELL 6b — SPEED PROFILE (~2 min): where does the 14% throughput loss between the smoke test and the real run come from? =================
# Times (a) building batches (tokenising text) and (b) the GPU forward/backward on a fixed batch, and reads the GPU clock/power. Read-only.
import subprocess
model_p, _, _ = build_model(tiny=TINY); model_p.train(); ps = PackedStream(tok, CFG_ARCH["micro_bs"], SEQ, COMMON["domain_w"], COMMON["min_words"], 999, SKIP); N = 20
t0 = time.time(); batches = [ps.next_batch() for _ in range(N)]; t_data = (time.time() - t0) / N
x = batches[0].to(device); g_ = torch.Generator(device=device).manual_seed(0); r_ = np.random.default_rng(0)
for _ in range(2):
    with amp_ctx(CFG_ARCH["amp"]): forward_loss(model_p, x, rng=r_, gen=g_).backward()
if device.type == "cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(N):
    with amp_ctx(CFG_ARCH["amp"]): forward_loss(model_p, x, rng=r_, gen=g_).backward()
if device.type == "cuda": torch.cuda.synchronize()
t_gpu = (time.time() - t0) / N; tokens = CFG_ARCH["micro_bs"] * SEQ
print(f"building one micro-batch of {tokens} tokens: {t_data*1000:.0f} ms | GPU forward+backward on it: {t_gpu*1000:.0f} ms | data/GPU = {t_data/t_gpu:.1%}")
print(f"  pure GPU speed {tokens/t_gpu/1e3:.1f}k tok/s; if data were NOT overlapped: {tokens/(t_gpu+t_data)/1e3:.1f}k tok/s")
try: print("  GPU clocks / power / temperature:", subprocess.run(["nvidia-smi", "--query-gpu=clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu", "--format=csv,noheader"], capture_output=True, text=True, timeout=20).stdout.strip())
except Exception as e: print("  nvidia-smi not available:", type(e).__name__)
print("  reading: data/GPU above ~10% -> the background prefetch (PREFETCH=True in the trainer cell) should recover most of the gap; if clocks are well below the maximum while power sits at its limit, the GPU is throttling and nothing in the code can fix it")
del model_p, ps, batches; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= CELL 7 — THE RUN: probes your real GPU speed, sizes the schedule to HOURS, trains, saves. Resumes automatically if the session is interrupted. =================
# Gate: within ~20 minutes the printed validation loss must be falling steadily. If it is flat or non-finite, stop and send me the log.
# For a short pilot first: run_arch(hours=0.4)   (then delete /kaggle/working/pretrain_modernbert/modernbert before the real run)
# v2 run: 3.5 GPU-hours. Make sure Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY are attached and Internet is ON, then Save & Run All: cell 9 pushes the model automatically.
info = run_arch()


In [ ]:
# ================= CELL 8 — export: the checkpoint for the Phase 2/3 notebooks + a loader module that handles all four architectures =================
LOADER_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
(OUT / "arch_loaders.py").write_text(LOADER_SRC); print("wrote", OUT / "arch_loaders.py")
sys.path.insert(0, str(OUT)); import importlib, arch_loaders; importlib.reload(arch_loaders)
p = OUT / ARCH / "final_model.pt"; assert p.exists(), "run cell 7 first"
bb, hidden, causal, ctx, fam = arch_loaders.load_backbone(str(p)); ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
ids = torch.randint(4, 1000, (2, 24)); mask = torch.ones_like(ids); mask[1, 16:] = 0
with torch.no_grad(): h = bb(input_ids=ids, attention_mask=mask).last_hidden_state
assert h.shape == (2, 24, hidden) and torch.isfinite(h).all()
print(f"{ARCH}: {p} | {ck['step']} steps | {sum(q.numel() for q in bb.parameters())/1e6:.1f}M backbone parameters | output {tuple(h.shape)} | causal={causal} | adapters: {arch_loaders.ADAPTER_TARGETS[fam]}")
print(f'Phase 3 registry ->  ARCHS["{ARCH}"] = dict(path="{p}", family="{fam}")')
print("\nDownload final_model.pt, arch_loaders.py and legal_bpe_32k_mask.json (vocabulary 32,001) from the output panel; the Phase 2/3 notebooks use all three.")


In [ ]:
# ================= CELL 9 — AUTOMATIC PUSH to a private Kaggle dataset right after the export, so the model can never be lost again =================
# Needs the Kaggle Secrets KAGGLE_USERNAME and KAGGLE_KEY attached to this notebook (Add-ons > Secrets) and Internet ON. It never asks questions, so it also works inside Save & Run All.
# Without credentials it only prints a warning: then DOWNLOAD final_model.pt, arch_loaders.py and legal_bpe_32k_mask.json from the output panel yourself before the session ends.
import hashlib, shutil, subprocess
DRY_RUN = False; INCLUDE_RESUME = False
SLUGS = {"gpt2": "cs781-phase1-v3-gpt2-decoder", "modernbert": "cs781-phase1-v3-encoder-modernbert", "t5": "cs781-phase1-v3-t5-encdec", "bilstm": "cs781-phase1-v3-bilstm-recurrent"}
KIND = {"gpt2": "decoder", "modernbert": "encoder", "t5": "enc-dec", "bilstm": "recurrent"}
DATASET_SLUG = SLUGS[ARCH]; src = OUT / ARCH
HAVE_CREDS = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
if not HAVE_CREDS:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient(); os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY"); HAVE_CREDS = True; print("using Kaggle secrets")
    except Exception as e: print(f"!!! NOT UPLOADED: no Kaggle credentials ({type(e).__name__}). Attach the secrets KAGGLE_USERNAME and KAGGLE_KEY, or download the three files from the output panel BEFORE the session ends.")
if HAVE_CREDS:
    OWNER = os.environ.get("KAGGLE_USERNAME", "oipranesh")
    stage = Path("/kaggle/working/upload_" + ARCH); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)
    files = {"final_model.pt": src / "final_model.pt", "log.jsonl": src / "log.jsonl", "arch_loaders.py": OUT / "arch_loaders.py", "legal_bpe_32k_mask.json": OUT / "legal_bpe_32k_mask.json", f"val_blocks_{SEQ}.pt": OUT / f"val_blocks_{SEQ}.pt"}
    if INCLUDE_RESUME: files["latest.pt"] = src / "latest.pt"
    for extra in ("old_vs_new_metrics.csv", "loss_curve.png", "surprise.html"):
        if (OUT / extra).exists(): files[extra] = OUT / extra
    if (OUT / "phase1_predictions").exists():                                             # flattened names: the Kaggle CLI skips sub-folders
        for p in (OUT / "phase1_predictions").glob("*"): files["phase1_" + p.name] = p
    missing = [n for n, p in files.items() if not p.exists()]; assert not missing, f"missing files: {missing} (run cells 7 and 8 first)"
    for name, p in files.items():
        try: os.link(p, stage / name)                                                      # hard link: no extra disk space
        except OSError: shutil.copy2(p, stage / name)

    def sha256(p, chunk=1 << 24):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(chunk), b""): h.update(b)
        return h.hexdigest()
    recs = [json.loads(l) for l in open(src / "log.jsonl")]; vals = [r for r in recs if "val" in r]; last = vals[-1]; ck = torch.load(stage / "final_model.pt", map_location="cpu", weights_only=False, mmap=True)
    tps = CFG_ARCH["micro_bs"] * SEQ * CFG_ARCH["accum"]
    manifest = {"architecture": ARCH, "model_config": ck["model_config"], "steps": ck["step"], "tokens": ck["step"] * tps, "final_validation": last["val"], "validation_history": [(v["step"], round(v["val"]["overall"], 4)) for v in vals],
                "tokenizer": "legal_bpe_32k + [MASK] (vocabulary 32001)", "recipe": {"peak_lr": CFG_ARCH["lr_peak"], "tokens_per_step": tps, "domain_weights": COMMON["domain_w"], "seq_len": SEQ}, "files": {n: {"bytes": (stage / n).stat().st_size, "sha256": sha256(stage / n)} for n in files}}
    (stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=1, default=str))
    (stage / "README.md").write_text(f"# CS781 Phase 1 v3 - {ARCH} ({KIND[ARCH]})\n\nSteps: {manifest['steps']} | tokens: {manifest['tokens']/1e6:.0f}M | final validation (loss per token, objective-specific): {json.dumps({k: round(v, 3) for k, v in last['val'].items()})}\n\n"
        f"Load: `import arch_loaders; backbone, hidden, causal, ctx, family = arch_loaders.load_backbone('final_model.pt')`. Tokenizer: legal_bpe_32k_mask.json (vocabulary 32,001). Integrity: see MANIFEST.json.\n")
    (stage / "dataset-metadata.json").write_text(json.dumps({"title": f"CS781 Phase 1 v3 - {ARCH} {KIND[ARCH]} ({manifest['steps']} steps)"[:50], "id": f"{OWNER}/{DATASET_SLUG}", "licenses": [{"name": "unknown"}]}, indent=1))
    tot = sum((stage / n).stat().st_size for n in list(files) + ["MANIFEST.json", "README.md", "dataset-metadata.json"])
    print(f"staged {len(files) + 3} files, {tot/2**30:.2f} GiB in {stage}:"); [print(f"  {n:34s} {m['bytes']/2**20:9.1f} MiB  sha256 {m['sha256'][:12]}") for n, m in manifest["files"].items()]
    print(f"\ntarget dataset: https://www.kaggle.com/datasets/{OWNER}/{DATASET_SLUG}  (private by default)")

    kaggle_bin = shutil.which("kaggle"); cmd_create = ["kaggle", "datasets", "create", "-p", str(stage)]; cmd_version = ["kaggle", "datasets", "version", "-p", str(stage), "-m", f"{ARCH} {manifest['steps']} steps; validation {last['val']['overall']:.3f}"]
    if DRY_RUN or not kaggle_bin: print("\nDRY RUN (or kaggle CLI not found): would run\n  ", " ".join(cmd_create), "\n  (or, if the dataset already exists)", " ".join(cmd_version), "\nSet DRY_RUN = False and run again to upload." if kaggle_bin else "\n`kaggle` CLI not found: use the manual download route.")
    else:
        chk = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=180)
        if chk.returncode != 0: print("credentials check FAILED (is Internet on? is this the legacy API key?):\n" + (chk.stdout + chk.stderr)[:400] + "\nNothing was uploaded. Re-run this cell to try again.")
        else:
            print(f"credentials accepted for '{OWNER}'. Uploading ...")
            r = subprocess.run(cmd_create, capture_output=True, text=True); out_ = (r.stdout + r.stderr).strip(); print("\n".join(l for l in out_.splitlines() if "%|" not in l)[-1500:])
            if r.returncode != 0 and "already" in out_.lower():
                print("dataset exists -> uploading a new version"); r = subprocess.run(cmd_version, capture_output=True, text=True); print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "%|" not in l)[-1500:])
            print("\nUPLOAD OK" if r.returncode == 0 else f"\nUPLOAD FAILED (exit {r.returncode}) - the staged folder {stage} is still there for the manual route")
            print(f'In later notebooks: Add Input > Your datasets > {DATASET_SLUG}   (the CJPE notebook finds final_model.pt and arch_loaders.py automatically)')
    del ck
